# A_6 — Systemic impact

**Input:** `Data/TYNDP 2024.xlsx` (CBA sheets), `Data/TYNDP 2020.xlsx`, `Data/TYNDP 2022.xlsx`
**Output:** Table 3

Projects the empirical failure and delay rates onto the project-level
cost-benefit analysis of the TYNDP 2024, for socio-economic welfare, integrated
renewable generation and avoided emissions. Benefits between the 2030 and 2040
horizons are linearly interpolated; weighted-average values are used throughout
rather than range extremes.

Failure probabilities are applied at the aggregate project level rather than
assuming that one delayed component blocks an entire project, and cancellation
losses are mitigated by a substitution rate. Both choices bias the estimate
downward, establishing a lower bound.

In [1]:
import pandas as pd
import numpy as np

In [2]:
file_path = "Data/TYNDP 2024.xlsx"

In [3]:

df_projects = pd.read_excel(file_path, sheet_name="Trans.Projects", header=1)

In [4]:
status_excel_name = (
    "Status ID\n"
    "1 : Under Consideration,\n"
    "2 : In Planning but not permitting,\n"
    "3 : In permitting,\n"
    "4 : Under Construction"
)

df_projects.rename(columns={
    status_excel_name: "Status",
    "Transfer capacity increase A-B (MW)": "Capacity_AB_MW",
    "Transfer capacity increase B-A (MW)": "Capacity_BA_MW",
    "Commissioning Year estimated by the promoter": "Planned_Comm_Year"
}, inplace=True)

cols_to_keep = ["Project ID", "Status", "Capacity_AB_MW", "Capacity_BA_MW", "Planned_Comm_Year"]
df_projects = df_projects[cols_to_keep].copy()

df_projects["Project ID"] = df_projects["Project ID"].astype(str).str.replace(".0", "", regex=False).str.strip()
df_projects["Planned_Comm_Year"] = df_projects["Planned_Comm_Year"].astype(str).str.extract(r'(\d{4})', expand=False).astype(float)

df_projects["Capacity_AB_MW"] = pd.to_numeric(df_projects["Capacity_AB_MW"], errors="coerce").fillna(0)
df_projects["Capacity_BA_MW"] = pd.to_numeric(df_projects["Capacity_BA_MW"], errors="coerce").fillna(0)
df_projects["Avg_Capacity_MW"] = (df_projects["Capacity_AB_MW"] + df_projects["Capacity_BA_MW"]) / 2

df_projects["Project ID"] = df_projects["Project ID"].astype(str).str.replace(".0", "", regex=False).str.strip()

df_projects = df_projects[~df_projects["Project ID"].isin(["nan", "None", ""])]
df_projects = df_projects.dropna(subset=["Project ID"])

df_projects["Planned_Comm_Year"] = df_projects["Planned_Comm_Year"].astype(str).str.extract(r'(\d{4})', expand=False).astype(float)

In [5]:

def extract_cba_data(sheet_name):
    print(f"Extracting data from {sheet_name}...")
    
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=[0, 1, 2])
    
    new_columns = []
    for col in df.columns:
        level_0 = str(col[0]).strip() 
        level_1 = str(col[1]).strip() 
        
        if "Unnamed" in level_1 or level_0 == level_1:
            new_columns.append(level_0)
        else:
            new_columns.append(f"{level_0}_{level_1}")
            
    df.columns = new_columns

    df["Project ID"] = df["Project ID"].astype(str).str.replace(".0", "", regex=False).str.strip()
    df = df[~df["Project ID"].isin(["nan", "None", ""])]
    df = df.dropna(subset=["Project ID"])
    df = df.drop_duplicates(subset=["Project ID"])
    
    target_columns = [
        "Project ID",
        "ΔSEW_weighted avg", 
        "ΔCO2_market_weighted avg", 
        "ΔRES_weighted avg", 
        "ΔRES_MW_weighted avg"
    ]
    
    existing_targets = [col for col in target_columns if col in df.columns]
    df_clean = df[existing_targets].copy()
    
    for col in df_clean.columns:
        if col != "Project ID":
            df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").fillna(0)
            
    return df_clean

df_2030 = extract_cba_data("2030NT")
df_2040 = extract_cba_data("2040NT")

df_2030 = df_2030.add_suffix("_2030").rename(columns={"Project ID_2030": "Project ID"})
df_2040 = df_2040.add_suffix("_2040").rename(columns={"Project ID_2040": "Project ID"})

print("Phase 2 complete! Dataframes are ready.")

Extracting data from 2030NT...
Extracting data from 2040NT...
Phase 2 complete! Dataframes are ready.


In [6]:
df_final = df_projects.merge(df_2030, on="Project ID", how="left")
df_final = df_final.merge(df_2040, on="Project ID", how="left")

df_final = df_final.fillna(0)

C:\Users\damia\AppData\Local\Temp\ipykernel_31160\3381478015.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_final = df_final.fillna(0)


In [7]:
# ==========================================
#  Impact of Cancellations (2030-2040)
# ==========================================
print("Calculating loss due to Cancellations only...")

cancel_probs = {1: 0.87, 2: 0.70, 3: 0.42, 4: 0.14}

df_final["Cancel_Prob"] = df_final["Status"].map(cancel_probs)

cancellation_losses = []

for index, row in df_final.iterrows():
    p_fail = row["Cancel_Prob"]
    t_plan = row["Planned_Comm_Year"]
    
    sew_30, sew_40 = row["ΔSEW_weighted avg_2030"], row["ΔSEW_weighted avg_2040"]
    co2_30, co2_40 = row["ΔCO2_market_weighted avg_2030"], row["ΔCO2_market_weighted avg_2040"]
    res_30, res_40 = row["ΔRES_weighted avg_2030"], row["ΔRES_weighted avg_2040"]
    
    loss_sew = 0
    loss_co2 = 0
    loss_res = 0
    
    if pd.isna(t_plan):
        t_plan = 2030 
        
    for year in range(int(t_plan), 2041):
        if year < 2030: continue 
        
        weight = (year - 2030) / 10
        b_sew = sew_30 + weight * (sew_40 - sew_30)
        b_co2 = co2_30 + weight * (co2_40 - co2_30)
        b_res = res_30 + weight * (res_40 - res_30)
        
        loss_sew += b_sew * p_fail
        loss_co2 += b_co2 * p_fail
        loss_res += b_res * p_fail
        
    cancellation_losses.append({'SEW': loss_sew, 'CO2': loss_co2, 'RES': loss_res})

df_losses = pd.DataFrame(cancellation_losses)
df_final["Loss_Canc_SEW_M€"] = df_losses['SEW'].values
df_final["Loss_Canc_CO2_ktonnes"] = -df_losses['CO2'].values
df_final["Loss_Canc_RES_GWh"] = df_losses['RES'].values

print("\n--- RESULTS: IMPACT OF CANCELLATIONS ONLY ---")
print(f"Total SEW Lost: {df_final['Loss_Canc_SEW_M€'].sum():,.0f} M€")
print(f"Total CO2 ktonnes Lost: {df_final['Loss_Canc_CO2_ktonnes'].sum():,.0f} ktonnes")
print(f"Total RES GWh Lost: {df_final['Loss_Canc_RES_GWh'].sum():,.0f} GWh")

Calculating loss due to Cancellations only...

--- RESULTS: IMPACT OF CANCELLATIONS ONLY ---
Total SEW Lost: 139,153 M€
Total CO2 ktonnes Lost: 380,187 ktonnes
Total RES GWh Lost: 2,218,718 GWh


### Impact of delay

In [8]:
files = {
    2020: "Data/TYNDP 2020.xlsx", 
    2022: "Data/TYNDP 2022.xlsx",
    2024: "Data/TYNDP 2024.xlsx"
}

data_frames = {}
for year, path in files.items():
    df = pd.read_excel(path, sheet_name="Trans.Projects", header=1)
    status_col = [c for c in df.columns if "Status ID" in str(c)][0]
    
    df = df[["Project ID", status_col]].copy()
    df.columns = ["Project ID", f"Status_{year}"]
    df["Project ID"] = df["Project ID"].astype(str).str.replace(".0", "", regex=False).str.strip()
    
    df = df[~df["Project ID"].isin(["nan", "", "None"])]
    data_frames[year] = df.drop_duplicates(subset=["Project ID"])


df_wide = data_frames[2020].merge(data_frames[2022], on="Project ID", how="outer")
df_wide = df_wide.merge(data_frames[2024], on="Project ID", how="outer")

df_wide = df_wide.sort_values("Project ID")


In [9]:
df_wide_2024 = df_wide.dropna(subset=["Status_2024"]).copy()

df_wide_2024 = df_wide_2024.reset_index(drop=True)

In [10]:
def count_years_in_status(row):
    current_status = row["Status_2024"]
    
    if row["Status_2022"] == current_status and row["Status_2020"] == current_status:
        return 4
    elif row["Status_2022"] == current_status:
        return 2
    else:
        return 0

df_wide_2024["Years_In_Status"] = df_wide_2024.apply(count_years_in_status, axis=1)

df_final_status = df_wide_2024[["Project ID", "Status_2024", "Years_In_Status"]]

print(df_final_status.head())

  Project ID Status_2024  Years_In_Status
0          1           4                2
1       1034           2                4
2       1039           1                4
3       1040           1                4
4       1041           1                4


In [11]:
avg_durations = {1: 2.2, 2: 3.1, 3: 5.0, 4: 3.8}

def calculate_expected_commissioning_time(row):
    status = row["Status_2024"]
    time_spent = row["Years_In_Status"]
    dur_current = avg_durations.get(status, 0)
    
    if time_spent >= dur_current:
        remaining_in_current = 1.0 
    else:
        remaining_in_current = (dur_current - time_spent) + 1.0
        
    remaining_phases = 0
    for s in range(int(status) + 1, 5):
        remaining_phases += avg_durations.get(s, 0)
        
    return remaining_in_current + remaining_phases

df_final_status = df_final_status.copy()

df_final_status["Expected_Years_To_Commission"] = df_final_status.apply(calculate_expected_commissioning_time, axis=1)

In [12]:
df_final_status["Computed_Commissioning_Year"] = 2024 + df_final_status["Expected_Years_To_Commission"]
print(df_final_status[["Project ID", "Status_2024", "Expected_Years_To_Commission", "Computed_Commissioning_Year"]].head())

  Project ID Status_2024  Expected_Years_To_Commission  \
0          1           4                           2.8   
1       1034           2                           9.8   
2       1039           1                          12.9   
3       1040           1                          12.9   
4       1041           1                          12.9   

   Computed_Commissioning_Year  
0                       2026.8  
1                       2033.8  
2                       2036.9  
3                       2036.9  
4                       2036.9  


# cost of delay

In [13]:
df_final_status["Project ID"] = df_final_status["Project ID"].astype(str)
df_final["Project ID"] = df_final["Project ID"].astype(str)

df_unified = df_final.merge(
    df_final_status[["Project ID", "Computed_Commissioning_Year"]], 
    on="Project ID", 
    how="left"
)

df_unified["Delay_Gap_Years"] = df_unified["Computed_Commissioning_Year"] - df_unified["Planned_Comm_Year"]

df_unified["Delay_Gap_Years"] = df_unified["Delay_Gap_Years"].clip(lower=0)

In [14]:
delay_losses = []

for index, row in df_unified.iterrows():
    t_plan = row["Planned_Comm_Year"]
    t_comp = row["Computed_Commissioning_Year"]
    
    loss_sew_d = 0
    loss_co2_d = 0
    loss_res_d = 0
    
    for year in range(int(t_plan), int(t_comp)):
        if year < 2030 or year > 2040: continue 
        
        weight = (year - 2030) / 10
        b_sew = row["ΔSEW_weighted avg_2030"] + weight * (row["ΔSEW_weighted avg_2040"] - row["ΔSEW_weighted avg_2030"])
        b_co2 = row["ΔCO2_market_weighted avg_2030"] + weight * (row["ΔCO2_market_weighted avg_2040"] - row["ΔCO2_market_weighted avg_2030"])
        b_res = row["ΔRES_weighted avg_2030"] + weight * (row["ΔRES_weighted avg_2040"] - row["ΔRES_weighted avg_2030"])
        
        loss_sew_d += b_sew
        loss_co2_d += b_co2
        loss_res_d += b_res
        
    delay_losses.append({'SEW': loss_sew_d, 'CO2': loss_co2_d, 'RES': loss_res_d})

df_delay = pd.DataFrame(delay_losses)
df_unified["Loss_Delay_SEW_M€"] = df_delay['SEW'].values
df_unified["Loss_Delay_CO2_ktonnes"] = -df_delay['CO2'].values
df_unified["Loss_Delay_RES_GWh"] = df_delay['RES'].values

print("\n--- RESULTS: IMPACT OF DELAYS ONLY ---")
print(f"Total SEW Lost due to Delays: {df_unified['Loss_Delay_SEW_M€'].sum():,.0f} M€")
print(f"Total CO2_ktonnes Lost due to Delays: {df_unified['Loss_Delay_CO2_ktonnes'].sum():,.0f} ")
print(f"Total RES_GWh Lost due to Delays: {df_unified['Loss_Delay_RES_GWh'].sum():,.0f} ")


--- RESULTS: IMPACT OF DELAYS ONLY ---
Total SEW Lost due to Delays: 60,227 M€
Total CO2_ktonnes Lost due to Delays: 222,075 
Total RES_GWh Lost due to Delays: 1,276,192 


# Total loss from 2030 to 2040 

In [15]:
# ==============================================================================
# UNMITIGATED DELAYS + MITIGATED CANCELLATIONS 
# ==============================================================================

raw_delay_sew = df_unified['Loss_Delay_SEW_M€'].sum()
raw_delay_co2 = df_unified['Loss_Delay_CO2_ktonnes'].sum()
raw_delay_res = df_unified['Loss_Delay_RES_GWh'].sum()

raw_cancel_sew = df_final['Loss_Canc_SEW_M€'].sum()
raw_cancel_co2 = df_final['Loss_Canc_CO2_ktonnes'].sum()
raw_cancel_res = df_final['Loss_Canc_RES_GWh'].sum()

sew_central = raw_delay_sew + (raw_cancel_sew*0.7)
co2_central = (raw_delay_co2 + (raw_cancel_co2*0.7)) / 1000
res_central = (raw_delay_res + (raw_cancel_res*0.7)) / 1000

sew_high = raw_delay_sew + (raw_cancel_sew*0.9)
co2_high = (raw_delay_co2 + (raw_cancel_co2*0.9)) / 1000
res_high = (raw_delay_res + (raw_cancel_res*0.9)) / 1000

sew_low = raw_delay_sew + (raw_cancel_sew*0.5)
co2_low = (raw_delay_co2 + (raw_cancel_co2*0.5)) / 1000
res_low = (raw_delay_res + (raw_cancel_res*0.5)) / 1000

print("--- COMBINED RISK ASSESSMENT: DELAYS & MITIGATED CANCELLATIONS (30% [10%-50%] SUB. RATE)  ---")
print(f"Total SEW Loss: {sew_central:,.0f} M€ [Substitution rate 50% {sew_low:,.0f} – 10% {sew_high:,.0f}]")
print(f"Total CO2 Loss: {co2_central:,.1f} Million tonnes [Substitution rate 50% {co2_low:,.1f} – 10% {co2_high:,.1f}]")
print(f"Total RES Loss: {res_central:,.0f} TWh [Substitution rate 50% {res_low:,.0f} – 10% {res_high:,.0f}]")

--- COMBINED RISK ASSESSMENT: DELAYS & MITIGATED CANCELLATIONS (30% [10%-50%] SUB. RATE)  ---
Total SEW Loss: 157,634 M€ [Substitution rate 50% 129,804 – 10% 185,465]
Total CO2 Loss: 488.2 Million tonnes [Substitution rate 50% 412.2 – 10% 564.2]
Total RES Loss: 2,829 TWh [Substitution rate 50% 2,386 – 10% 3,273]


# Cost of cancellation and missing planning at 2040

In [16]:
total_sew_2040 = df_unified["ΔSEW_weighted avg_2040"].sum()
total_co2_2040 = df_unified["ΔCO2_market_weighted avg_2040"].sum() # in ktonnes
total_res_2040 = df_unified["ΔRES_weighted avg_2040"].sum()        # in GWh

total_gw = df_unified["Avg_Capacity_MW"].sum() / 1000

sew_per_gw = total_sew_2040 / total_gw
co2_per_gw = total_co2_2040 / total_gw
res_per_gw = total_res_2040 / total_gw

permanent_annual_sew_loss = sew_per_gw * 28
permanent_annual_co2_loss = -(co2_per_gw * 28) / 1000  # Convert ktonnes to Million tonnes
permanent_annual_res_loss = (res_per_gw * 28) / 1000  # Convert GWh to TWh

print("--- STRATEGIC ASSESSMENT: STRUCTURAL DEFICIT (Post-2040) ---")
print(f"Total capacity in dataset (GW): {total_gw:,.2f} GW")
print(f"Annual SEW Loss from 28 GW gap: {permanent_annual_sew_loss:,.2f} Million €/year")
print(f"Annual CO2 Loss from 28 GW gap: {permanent_annual_co2_loss:,.2f} Million tonnes/year")
print(f"Annual RES Loss from 28 GW gap: {permanent_annual_res_loss:,.2f} TWh/year")


--- STRATEGIC ASSESSMENT: STRUCTURAL DEFICIT (Post-2040) ---
Total capacity in dataset (GW): 140.66 GW
Annual SEW Loss from 28 GW gap: 6,577.53 Million €/year
Annual CO2 Loss from 28 GW gap: 11.48 Million tonnes/year
Annual RES Loss from 28 GW gap: 86.45 TWh/year


In [17]:
cancel_probs = {1: 0.87, 2: 0.70, 3: 0.42, 4: 0.14}
df_unified["Cancel_Prob"] = df_unified["Status"].map(cancel_probs)

df_unified["Permanent_Annual_Loss_SEW_M€"] = df_unified["ΔSEW_weighted avg_2040"] * df_unified["Cancel_Prob"]
df_unified["Permanent_Annual_Loss_CO2_ktonnes"] = df_unified["ΔCO2_market_weighted avg_2040"] * df_unified["Cancel_Prob"]
df_unified["Permanent_Annual_Loss_RES_GWh"] = df_unified["ΔRES_weighted avg_2040"] * df_unified["Cancel_Prob"]

total_perm_sew = df_unified["Permanent_Annual_Loss_SEW_M€"].sum()
total_perm_co2 = -df_unified["Permanent_Annual_Loss_CO2_ktonnes"].sum()
total_perm_res = df_unified["Permanent_Annual_Loss_RES_GWh"].sum()

print("\n--- RESULTS: ANNUAL PERMANENT LOSS (2040+) FROM CANCELLED PROJECTS ---")
print(f"Annual SEW Permanent Loss: {total_perm_sew:,.0f} M€/year")
print(f"Annual CO2 Permanent Loss: {total_perm_co2:,.0f} ktonnes/year")
print(f"Annual RES Permanent Loss: {total_perm_res:,.0f} GWh/year")


--- RESULTS: ANNUAL PERMANENT LOSS (2040+) FROM CANCELLED PROJECTS ---
Annual SEW Permanent Loss: 20,312 M€/year
Annual CO2 Permanent Loss: 36,765 ktonnes/year
Annual RES Permanent Loss: 314,020 GWh/year


In [18]:

total_annual_sew = permanent_annual_sew_loss + total_perm_sew
total_annual_co2 = permanent_annual_co2_loss + (total_perm_co2 / 1000)
total_annual_res = permanent_annual_res_loss + (total_perm_res / 1000)

sew_central = total_annual_sew * 0.7
sew_low = total_annual_sew * 0.5
sew_high = total_annual_sew * 0.9

co2_central = total_annual_co2 * 0.7
co2_low = total_annual_co2 * 0.5
co2_high = total_annual_co2 * 0.9

res_central = total_annual_res * 0.7
res_low = total_annual_res * 0.5
res_high = total_annual_res * 0.9

print("--- TOTAL ANNUAL SYSTEMIC LOSS POST-2040 ---")
print(f"Total Annual SEW Loss: {sew_central:,.0f} M€/year [Substitution rate 50% {sew_low:,.0f} – 10% {sew_high:,.0f}]")
print(f"Total Annual CO2 Loss: {co2_central:,.1f} Million tonnes/year [Substitution rate 50% {co2_low:,.1f} – 10% {co2_high:,.1f}]")
print(f"Total Annual RES Loss: {res_central:,.0f} TWh/year [Substitution rate 50% {res_low:,.0f} – 10% {res_high:,.0f}]")

--- TOTAL ANNUAL SYSTEMIC LOSS POST-2040 ---
Total Annual SEW Loss: 18,823 M€/year [Substitution rate 50% 13,445 – 10% 24,200]
Total Annual CO2 Loss: 33.8 Million tonnes/year [Substitution rate 50% 24.1 – 10% 43.4]
Total Annual RES Loss: 280 TWh/year [Substitution rate 50% 200 – 10% 360]
